# MEDREx — Method 3: Bio_ClinicalBERT Fine-Tuning
**CSC-590 Masters Project, CSUDH Spring 2026**

Fine-tuning `emilyalsentzer/Bio_ClinicalBERT` on 9,523 TCGA pathology reports → 33 cancer type classes.

### Before running:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload to Google Drive `My Drive/MEDREx/`:
   - `TCGA_Reports.csv`
   - `tcga_patient_to_cancer_type.csv`
3. Run cells **one by one in order**

In [ ]:
# CELL 1 — Check GPU
# Expected: GPU available: True  |  GPU name: Tesla T4
# If False → Runtime → Change runtime type → T4 GPU
import torch
print(f'GPU available  : {torch.cuda.is_available()}')
print(f'GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — change runtime!"}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# CELL 2 — Install packages
# Colab has torch pre-installed. We only need transformers.
!pip install transformers scikit-learn -q
print('Packages ready.')

In [ ]:
# CELL 3 — Mount Google Drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

REPORTS_PATH = '/content/drive/MyDrive/MEDREx/TCGA_Reports.csv'
LABELS_PATH  = '/content/drive/MyDrive/MEDREx/tcga_patient_to_cancer_type.csv'

print('Loading reports...')
df_reports = pd.read_csv(REPORTS_PATH)
print(f'  Reports loaded : {len(df_reports):,} rows')

print('Loading labels...')
df_labels = pd.read_csv(LABELS_PATH)
print(f'  Labels loaded  : {len(df_labels):,} rows')

# Extract patient_id from filename
df_reports['patient_id'] = df_reports['patient_filename'].apply(lambda x: x.split('.')[0])
df_reports = df_reports[['patient_id', 'text']]
df_reports.index = df_reports['patient_id'].values
df_labels.index  = df_labels['patient_id'].values

# Merge
df_reports['cancer_type'] = df_labels.loc[df_reports.index, 'cancer_type']
df_reports = df_reports.dropna(subset=['cancer_type'])

print(f'\nMerged dataset : {df_reports.shape}')
print(f'Cancer types   : {df_reports["cancer_type"].nunique()} unique types')
print(df_reports['cancer_type'].value_counts())

In [ ]:
# CELL 4 — Build label encoder (cancer type string → number)
# BERT needs integer labels, not strings like 'KIRC' or 'BRCA'
import json
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_reports['label'] = le.fit_transform(df_reports['cancer_type'])

NUM_CLASSES = len(le.classes_)
label_map   = {int(i): cls for i, cls in enumerate(le.classes_)}

print(f'Number of classes : {NUM_CLASSES}')
print('Label mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i:2d} -> {cls}')

In [ ]:
# CELL 5 — Stratified 70/30 split (same random_state=42 as local Methods 1, 2, 4)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_reports['text'],
    df_reports['label'],
    test_size=0.3,
    random_state=42,
    stratify=df_reports['label']
)

print(f'Train : {len(X_train):,} reports')
print(f'Test  : {len(X_test):,}  reports')
print(f'Classes in train : {y_train.nunique()}')
print(f'Classes in test  : {y_test.nunique()}')

In [ ]:
# CELL 6 — Load Bio_ClinicalBERT tokenizer
# Bio_ClinicalBERT is trained on MIMIC-III clinical notes
# (real hospital records, very similar to our TCGA pathology reports)
from transformers import AutoTokenizer

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

# Show example tokenization
sample = str(X_train.iloc[0])[:200]
tokens = tokenizer(sample, truncation=True, max_length=512)
print(f'\nSample text    : {sample}')
print(f'Token count    : {len(tokens["input_ids"])}')
print(f'Max allowed    : 512 (longer reports are truncated at 512 tokens)')

In [ ]:
# CELL 7 — Tokenize all reports
# Converts every report text -> token IDs that BERT can read
# padding=True: all reports padded to same length within each batch
# truncation=True: reports longer than 512 tokens are cut off
MAX_LEN    = 512
BATCH_SIZE = 16   # fits in T4 16GB VRAM

print(f'Tokenizing {len(X_train):,} training reports...')
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors='pt'
)
print('Train tokenization done.')

print(f'Tokenizing {len(X_test):,} test reports...')
test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors='pt'
)
print('Test tokenization done.')

print(f'\nTrain input_ids shape : {train_encodings["input_ids"].shape}')
print(f'Test  input_ids shape : {test_encodings["input_ids"].shape}')

In [ ]:
# CELL 8 — Create PyTorch Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class CancerReportDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels.values, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = CancerReportDataset(train_encodings, y_train)
test_dataset  = CancerReportDataset(test_encodings,  y_test)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Test  batches : {len(test_loader)}')
print(f'Batch size    : {BATCH_SIZE}')

In [ ]:
# CELL 9 — Load Bio_ClinicalBERT with classification head
# AutoModelForSequenceClassification adds a linear layer on top:
# BERT (768 hidden) -> Linear(768, NUM_CLASSES) -> softmax -> cancer type
from transformers import AutoModelForSequenceClassification

print(f'Loading: {MODEL_NAME}')
print(f'Adding classification head: 768 hidden -> {NUM_CLASSES} cancer classes')

model  = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nDevice           : {device}')
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')
print(f'Model size       : ~{total_params * 4 / 1e6:.0f} MB')

In [ ]:
# CELL 10 — Optimizer and learning rate scheduler
# AdamW + linear warmup is the standard recipe for BERT fine-tuning
from transformers import AdamW, get_linear_schedule_with_warmup

EPOCHS       = 4       # 4 epochs ~ 40-60 min on T4
LR           = 2e-5    # standard BERT fine-tuning learning rate
WARMUP_STEPS = 100     # ramp up LR slowly at start to stabilise training

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f'Epochs         : {EPOCHS}')
print(f'Learning rate  : {LR}')
print(f'Steps/epoch    : {len(train_loader)}')
print(f'Total steps    : {total_steps}')
print(f'Warmup steps   : {WARMUP_STEPS}')

In [ ]:
# CELL 11 — Fine-tuning training loop
# This is where BERT learns our 33 cancer types.
# Watch: loss decreases each epoch, accuracy increases.
# Best model is auto-saved to Google Drive after every improvement.
import time
from sklearn.metrics import accuracy_score

best_val_acc   = 0.0
best_model_dir = '/content/drive/MyDrive/MEDREx/method3_best_model'

print('=' * 60)
print('  Starting fine-tuning...')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    # ── Training phase
    model.train()
    total_loss, train_preds, train_labels = 0, [], []
    t0 = time.time()

    for batch_num, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        train_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        if batch_num % 50 == 0:
            print(f'  Epoch {epoch} | Batch {batch_num}/{len(train_loader)} | Loss={loss.item():.4f}')

    train_acc  = accuracy_score(train_labels, train_preds)
    avg_loss   = total_loss / len(train_loader)
    epoch_mins = (time.time() - t0) / 60

    # ── Validation phase
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            out   = model(input_ids=batch['input_ids'].to(device),
                          attention_mask=batch['attention_mask'].to(device))
            val_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            val_labels.extend(batch['labels'].numpy())

    val_acc = accuracy_score(val_labels, val_preds)

    print(f'\n{"="*60}')
    print(f'  EPOCH {epoch}/{EPOCHS} COMPLETE')
    print(f'  Time          : {epoch_mins:.1f} min')
    print(f'  Train Loss    : {avg_loss:.4f}')
    print(f'  Train Accuracy: {train_acc*100:.2f}%')
    print(f'  Test  Accuracy: {val_acc*100:.2f}%')
    print(f'{"="*60}\n')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(best_model_dir, exist_ok=True)
        model.save_pretrained(best_model_dir)
        tokenizer.save_pretrained(best_model_dir)
        with open(f'{best_model_dir}/label_map.json', 'w') as f:
            json.dump(label_map, f)
        print(f'  NEW BEST MODEL saved to Google Drive')
        print(f'  Best accuracy : {best_val_acc*100:.2f}%\n')

print('Fine-tuning complete!')

In [ ]:
# CELL 12 — Final evaluation with full classification report
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForSequenceClassification

print('Loading best saved model...')
best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
best_model = best_model.to(device)
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        out = best_model(input_ids=batch['input_ids'].to(device),
                         attention_mask=batch['attention_mask'].to(device))
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

pred_names  = le.inverse_transform(all_preds)
label_names = le.inverse_transform(all_labels)

final_acc  = accuracy_score(all_labels, all_preds)
final_f1_w = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
final_f1_m = f1_score(all_labels, all_preds, average='macro',    zero_division=0)

print(f'\n{"="*60}')
print(f'  METHOD 3 — Bio_ClinicalBERT FINAL RESULTS')
print(f'{"="*60}')
print(f'  Test Accuracy : {final_acc*100:.2f}%')
print(f'  F1 Weighted   : {final_f1_w*100:.2f}%')
print(f'  F1 Macro      : {final_f1_m*100:.2f}%')
print(f'{"="*60}')
print(f'\n  COMPARISON:')
print(f'  Cedars-Sinai (BoW + LR)     : 95.31%')
print(f'  Method 1 (TF-IDF + LR)      : 96.36%')
print(f'  Method 2 (TF-IDF + SVM)     : ~96%')
print(f'  Method 4 (Embedding k-NN)   : 90.13%')
print(f'  Method 3 (BERT fine-tuned)  : {final_acc*100:.2f}%  <- NEW')
print(f'\nFull classification report:')
print(classification_report(label_names, pred_names, zero_division=0))

## After training is complete — Download model to your laptop

The trained model is saved in your Google Drive at:
```
My Drive/MEDREx/method3_best_model/
├── config.json
├── pytorch_model.bin       (~440 MB — the trained weights)
├── tokenizer_config.json
├── vocab.txt
├── special_tokens_map.json
└── label_map.json          (number → cancer type name)
```

**Download steps:**
1. Go to [drive.google.com](https://drive.google.com)
2. Open `MEDREx` → `method3_best_model` folder
3. Right-click the folder → **Download** (zips it, ~450MB)
4. Extract to: `MastersProject/demo/ProjectCode/method3_best_model/`
5. Run locally:
```bash
python demo/ProjectCode/method3_evaluate_local.py
```